In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd()

while not (ROOT / "spark").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

Project root: /Users/libowen/SD3rd/RP/code/prediction


## running time: 263m17s

In [ ]:
from argparse import Namespace
import torch
from torch.utils.data import DataLoader
import random
import numpy as np

from spark.streaming_dataset import TurbineStreamingDataset
from spark.train_utils_streaming import train_streaming, test_streaming

from ml.models.lstm import LSTM


args = Namespace(
    epochs=30,
    lr=0.001,
    batch_size=512,
    device=(
        "cuda" if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available()
        else "cpu"
    ),
    num_lags=24,
    seed=0,
)


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def main():

    set_seed(args.seed)

    static_cols = ("Capacity_kw", "age")

    train_dataset = TurbineStreamingDataset(
        data_dir = str(ROOT / "spark" / "processed_data" / "train"),
        static_cols=static_cols,
        shuffle=True,
    )

    val_dataset = TurbineStreamingDataset(
        data_dir = str(ROOT / "spark" / "processed_data" / "val"),
        static_cols=static_cols,
        shuffle=False,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        num_workers=0,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        num_workers=0,
    )

    sample_x, sample_y = next(iter(train_loader))

    input_dim = sample_x.shape[-1]
    out_dim = sample_y.shape[-1]

    model = LSTM(
        input_dim=input_dim,
        lstm_hidden_size=128,
        num_lstm_layers=1,
        layer_units=[128],
        num_outputs=out_dim,
        matrix_rep=True,
    )

    device = torch.device(args.device)

    best_model = train_streaming(
        model,
        train_loader,
        val_loader,
        device,
        epochs=args.epochs,
        lr=args.lr,
    )

    val_metrics = test_streaming(best_model, val_loader, device)

    print("\nFinal Validation:", val_metrics)


if __name__ == "__main__":
    main()

/Users/libowen/Library/Python/3.10/lib/python/site-packages/torch/nn/modules/rnn.py:82: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "



Epoch 1/30
Train → MSE:0.0057 RMSE:0.0756 MAE:0.0393 R2:0.5943
Val   → MSE:0.0053 RMSE:0.0729 MAE:0.0346 R2:0.6914

Epoch 2/30
Train → MSE:0.0044 RMSE:0.0663 MAE:0.0339 R2:0.6877
Val   → MSE:0.0046 RMSE:0.0678 MAE:0.0350 R2:0.7334

Epoch 3/30
Train → MSE:0.0040 RMSE:0.0632 MAE:0.0317 R2:0.7165
Val   → MSE:0.0059 RMSE:0.0767 MAE:0.0358 R2:0.6583

Epoch 4/30
Train → MSE:0.0038 RMSE:0.0615 MAE:0.0308 R2:0.7315
Val   → MSE:0.0054 RMSE:0.0734 MAE:0.0403 R2:0.6874

Epoch 5/30
Train → MSE:0.0037 RMSE:0.0611 MAE:0.0302 R2:0.7351
Val   → MSE:0.0046 RMSE:0.0678 MAE:0.0353 R2:0.7330

Epoch 6/30
Train → MSE:0.0036 RMSE:0.0596 MAE:0.0293 R2:0.7476
Val   → MSE:0.0044 RMSE:0.0662 MAE:0.0412 R2:0.7452

Epoch 7/30
Train → MSE:0.0035 RMSE:0.0594 MAE:0.0291 R2:0.7494
Val   → MSE:0.0039 RMSE:0.0623 MAE:0.0296 R2:0.7745

Epoch 8/30
Train → MSE:0.0035 RMSE:0.0589 MAE:0.0291 R2:0.7535
Val   → MSE:0.0038 RMSE:0.0620 MAE:0.0310 R2:0.7770

Epoch 9/30
Train → MSE:0.0035 RMSE:0.0590 MAE:0.0288 R2:0.7528
Val   → 